In [0]:
# Configuration
dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")
bronze_table = f"{catalog}.bronze.audio_raw"

In [0]:
# Initialize Database
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.bronze")
spark.sql(f"USE {catalog}.bronze")

In [0]:
# Create a Volume to store checkpoints and other non-tabular files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")
print(f"Volume created: /Volumes/{catalog}/bronze/checkpoints")

In [0]:
# Read all MP3s from S3 in parallel
raw_files_df = (spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true") 
    .option("pathGlobFilter", "*.mp3")   
    .load("s3://fma-data-bucket/raw_audio/"))

optimized_bronze_df = (raw_files_df
    .withColumn("track_id", regexp_extract(col("path"), r"(\d+)\.mp3$", 1).cast("int"))
    .withColumn("ingested_at", current_timestamp())
    .select("track_id", "content", "path", "length", "ingested_at"))

# Write to Delta Table for permanent storage
(optimized_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table))

print(f"Success! {spark.table(bronze_table).count()} tracks cataloged in Bronze.")

In [0]:
spark.table("fma_catalog.bronze.audio_events").printSchema()

In [0]:
# dbutils.fs.rm(checkpoint_path, recurse=True)
# print(f"CheckApoint cleared at {checkpoint_path}")